# template

> Row templates, transforms and the row builder: how a state and one typed question become one token row with an anchor in front of every option.

In [ ]:
#| default_exp template

In [ ]:
#| export
import copy, hashlib, importlib, json, random, re, string, warnings
from pathlib import Path

from fastcore.basics import patch, store_attr

from sysone.core import *

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail, test_ne
from transformers import AutoTokenizer
import json

One question becomes one row: the instructions, then every option behind its own anchor token, then the state. Laya's row on ModernBERT's tokenizer reads

```
[CLS] choice question: {instructions} [SEP] [MASK] opt₁ [MASK] opt₂ … [SEP] {state} [SEP]
```

and the encoder's vector at each `[MASK]` is what the head scores. This notebook builds that row in three pieces: *transforms* normalise a record, a `RowTemplate` says what the row looks like and how many tokens each part may use, and a `RowBuilder` applies both with a tokenizer, segment by segment, so every anchor position is known exactly. The tests use the fixture tokenizer, a small BPE shaped like ModernBERT's (same special tokens, same `[MASK]` that swallows the space before it):

In [ ]:
tok = AutoTokenizer.from_pretrained("fixtures/tiny-modernbert")
recs = [json.loads(l) for l in Path("fixtures/typed_decisions_tiny.jsonl").read_text().splitlines()]
rec = recs[0]
tok.cls_token, tok.sep_token, tok.mask_token, tok(" [MASK]", add_special_tokens=False)["input_ids"]

('[CLS]', '[SEP]', '[MASK]', [4])

What happens to a record, in order:

```{mermaid}
flowchart LR
    R["record<br/>state · questions · gold"] --> T["transforms<br/>Field, Regex, Truncate, Shuffle, Image"]
    T --> S["segments of each question's row<br/>specials · text runs · options · state"]
    S --> K["one tokenizer call<br/>for the whole case"]
    K --> B["budgets<br/>options, then question, then state"]
    B --> L["layout in template order,<br/>each anchor's position recorded"]
    L --> ROW["Row(ids, markers, qtype, target)"]
```

The state is tokenized once however many questions ask about it, and every segment is tokenized on its own, so the position of every anchor is known exactly rather than searched for afterwards.

## Transforms

A transform is a small callable that takes a record and returns a record, plus `to_json()` so the list can be written into `sysone.json` and rebuilt at inference: a row at inference must be the row at training, byte for byte. Records here are whole cases (`state`, `questions`, optional `gold`), so one transform sees a state once, however many questions ask about it. `train_only=True` marks augmentation that must not run at inference.

In [ ]:
#| export
_TRANSFORMS = {}

def transform(name:str):
    "Register a `Transform` subclass under `name` so exports that use it stay loadable"
    def _inner(cls):
        cls.name = name
        _TRANSFORMS[name] = cls
        return cls
    return _inner

class Transform:
    "Base class: `__call__(record) -> record`, `to_json()`, and `train_only` for augmentation"
    name, train_only = None, False
    def __call__(self, record:dict) -> dict: return record
    def to_json(self) -> dict:
        return {"name": self.name, **{k: v for k, v in vars(self).items() if not k.startswith("_")}}
    def __repr__(self):
        args = ", ".join(f"{k}={v!r}" for k, v in self.to_json().items() if k != "name")
        return f"{self.name}({args})"
    @staticmethod
    def from_json(d:dict):
        "The transform described by `d` (a `to_json()` result)"
        d = dict(d); name = d.pop("name")
        if name not in _TRANSFORMS: raise ValueError(f"unknown transform {name!r}; register it with @sysone.transform({name!r})")
        return _TRANSFORMS[name](**d)

def tfms_to_json(tfms) -> list: return [t.to_json() for t in tfms or []]
def tfms_from_json(ds) -> list: return [Transform.from_json(d) for d in ds or []]

def apply_tfms(tfms, record:dict, train:bool=False) -> dict:
    "Run `tfms` over a record; `train_only` ones only when `train`"
    for t in tfms or []:
        if t.train_only and not train: continue
        record = t(record)
    return record

Most transforms act on text in three places, the state, each question's instructions and each option's description, and never on the labels, which are answer keys. `_map_text` walks a structured state and applies a function to every string leaf:

In [ ]:
#| export
def _map_text(obj, f):
    if isinstance(obj, str): return f(obj)
    if isinstance(obj, dict): return {k: _map_text(v, f) for k, v in obj.items()}
    if isinstance(obj, list): return [_map_text(v, f) for v in obj]
    return obj

def _map_record(record, f, state=True, instructions=True, options=True):
    "A copy of `record` with `f` applied to the chosen text fields"
    r = dict(record)
    if state: r["state"] = _map_text(r["state"], f)
    if instructions or options:
        qs = {}
        for qid, q in r["questions"].items():
            q = dict(q)
            if instructions and isinstance(q.get("instructions"), str): q["instructions"] = f(q["instructions"])
            crit = q.get("criteria")
            if options and isinstance(crit, dict): q["criteria"] = {k: _map_text(v, f) for k, v in crit.items()}
            elif options and isinstance(crit, list) and q.get("type") == "score": q["criteria"] = [_map_text(v, f) for v in crit]
            qs[qid] = q
        r["questions"] = qs
    return r

In [ ]:
#| export
@transform("Regex")
class Regex(Transform):
    "Replace `pattern` with `repl` in the state, the instructions and the option descriptions"
    def __init__(self, pattern:str, repl:str="", state:bool=True, instructions:bool=True, options:bool=True):
        store_attr(); self._re = re.compile(pattern)
    def __call__(self, record):
        return _map_record(record, lambda s: self._re.sub(self.repl, s), self.state, self.instructions, self.options)

@transform("Lower")
class Lower(Transform):
    "Lower-case the state, the instructions and the option descriptions"
    def __init__(self, state:bool=True, instructions:bool=True, options:bool=True): store_attr()
    def __call__(self, record): return _map_record(record, str.lower, self.state, self.instructions, self.options)

In [ ]:
r = Regex(r"https?://\S+", "<url>")({"state": {"page": "see https://x.com/a now"}, "questions": {}})
test_eq(r["state"], {"page": "see <url> now"})
low = Lower()(rec)
test_eq(low["questions"]["action"]["instructions"], rec["questions"]["action"]["instructions"].lower())
test_eq(list(low["questions"]["action"]["criteria"]), list(rec["questions"]["action"]["criteria"]))

`Field` renders a structured state to text with a format string (nested keys use format's own `{page[title]}` syntax, lists and dicts are written as JSON); `Truncate` cuts the state text to a character budget, before tokenization, which is cheaper than letting the tokenizer see a 50 kB page:

In [ ]:
#| export
class _JSONFormatter(string.Formatter):
    def format_field(self, value, spec):
        if isinstance(value, (list, dict)) and not spec: return json.dumps(value, ensure_ascii=False)
        return super().format_field(value, spec)

@transform("Field")
class Field(Transform):
    "Render a dict state to text with a format string, e.g. `Field(state='{page[title]}\\n{page[text]}')`"
    def __init__(self, state:str): store_attr()
    def __call__(self, record):
        s = record["state"]
        if not isinstance(s, dict): return record
        return {**record, "state": _JSONFormatter().vformat(self.state, (), s)}

@transform("Truncate")
class Truncate(Transform):
    "Cut the state text to `state_chars` characters, keeping the start (`side='right'`) or the end"
    def __init__(self, state_chars:int, side:str="right"): store_attr()
    def __call__(self, record):
        s = record["state"]
        if not isinstance(s, str): s = render_state(s)
        if len(s) > self.state_chars: s = s[:self.state_chars] if self.side == "right" else s[-self.state_chars:]
        return {**record, "state": s}

In [ ]:
st = {"page": {"title": "Shop", "text": "Shirts"}, "recent_actions": ["open", "search"]}
test_eq(Field("{page[title]}\n{page[text]}\nrecent actions: {recent_actions}")({"state": st, "questions": {}})["state"],
        'Shop\nShirts\nrecent actions: ["open", "search"]')
test_eq(Truncate(5)({"state": "abcdefgh", "questions": {}})["state"], "abcde")
test_eq(Truncate(5, side="left")({"state": "abcdefgh", "questions": {}})["state"], "defgh")

`Shuffle` reorders the options of choice questions, a training-time augmentation so the head cannot learn option positions. Gold answers are keyed by label, so they follow the options for free. Score levels are ordered and nouls have a fixed false/true order, so both are left alone. The permutation depends on the seed, the epoch (set by the `OptionShuffle` callback in `sysone.learner`) and the case, so a run is reproducible and every epoch sees a new order:

In [ ]:
#| export
@transform("Shuffle")
class Shuffle(Transform):
    "Shuffle the options of choice questions (training only by default)"
    def __init__(self, options:bool=True, train_only:bool=True, seed:int=0): store_attr(); self._epoch = 0
    def set_epoch(self, epoch:int): self._epoch = int(epoch)
    def __call__(self, record):
        if not self.options: return record
        rng = random.Random(f"{self.seed}:{self._epoch}:{record.get('id', render_state(record['state'])[:200])}")
        qs = {}
        for qid, q in record["questions"].items():
            crit = q.get("criteria")
            if q.get("type") == "choice" and crit:
                items = list(crit.items()) if isinstance(crit, dict) else [(c, None) for c in crit]
                rng.shuffle(items)
                q = {**q, "criteria": dict(items)}
            qs[qid] = q
        return {**record, "questions": qs}

In [ ]:
sh = Shuffle(seed=1)
a = sh(rec)["questions"]["action"]["criteria"]
test_eq(sorted(a), sorted(rec["questions"]["action"]["criteria"]))
test_eq(sh(rec)["questions"]["action"]["criteria"], a)                  # same epoch, same order
sh.set_epoch(1)
test_ne(list(sh(rec)["questions"]["action"]["criteria"]), list(a))     # a new epoch, a new order
test_eq(sh(rec)["questions"]["urgency"], rec["questions"]["urgency"])   # score levels keep their order

`Image` is the multimodal transform: it takes the image out of a state such as `{"text": ..., "image": "shots/0412.png"}`, loads it (a path, bytes or a PIL image), resizes it so its long side is `side` pixels, and keeps it in `record["images"]` for the processor, leaving the text as the state. It needs pillow (`sysone[multimodal]`), imported only when called.

In [ ]:
#| export
def load_image(img, side:int|None=None):
    "A PIL RGB image from a path, a base64 data URL, bytes or a PIL image, resized so its long side is `side`"
    from PIL import Image as PILImage
    import io, base64
    if isinstance(img, str) and img.startswith("data:image/"): img = PILImage.open(io.BytesIO(base64.b64decode(img.split(",", 1)[1])))
    elif isinstance(img, (str, Path)): img = PILImage.open(img)
    elif isinstance(img, (bytes, bytearray)): img = PILImage.open(io.BytesIO(img))
    elif isinstance(img, dict) and "bytes" in img: img = PILImage.open(io.BytesIO(img["bytes"]))
    img = img.convert("RGB")
    if side and max(img.size) != side:
        w, h = img.size; s = side / max(w, h)
        img = img.resize((max(1, round(w * s)), max(1, round(h * s))), PILImage.BICUBIC)
    return img

@transform("Image")
class Image(Transform):
    "Move the image at `state[key]` into `record['images']`, resized to `side` px on the long side"
    def __init__(self, key:str="image", side:int|None=1024): store_attr()
    def __call__(self, record):
        s = record["state"]
        if not isinstance(s, dict): return record
        if s.get(self.key) is None:        # no image: a state that is only text reads as that text all the same
            return {**record, "state": s["text"]} if set(s) - {self.key} == {"text"} else record
        s = dict(s); img = s.pop(self.key)
        text = s.pop("text") if set(s) == {"text"} else (s if s else "")
        return {**record, "state": text, "images": [load_image(img, self.side)]}

In [ ]:
test_eq(Image()({"state": "no image", "questions": {}})["state"], "no image")
test_eq(Image()({"state": {"text": "only text"}, "questions": {}})["state"], "only text")
import base64, io
from PIL import Image as PILImage
buf = io.BytesIO(); PILImage.new("RGB", (8, 4), "red").save(buf, "PNG")
test_eq(load_image("data:image/png;base64," + base64.b64encode(buf.getvalue()).decode(), side=16).size, (16, 8))
js = tfms_to_json([Field("{a}"), Regex(r"https?://\S+", "<url>"), Truncate(1500), Shuffle()])
js

[{'name': 'Field', 'state': '{a}'},
 {'name': 'Regex',
  'pattern': 'https?://\\S+',
  'repl': '<url>',
  'state': True,
  'instructions': True,
  'options': True},
 {'name': 'Truncate', 'state_chars': 1500, 'side': 'right'},
 {'name': 'Shuffle', 'options': True, 'train_only': True, 'seed': 0}]

In [ ]:
back = tfms_from_json(js)
test_eq([t.to_json() for t in back], js)
test_eq(apply_tfms(back, {"state": {"a": "go to http://x.y"}, "questions": {}})["state"], "go to <url>")

### Side streams

`Image` moves an image out of the state because a multimodal encoder reads it differently from text. The same holds for any input that has its own tokenizer or its own encoder: a protein sequence for a protein language model, a SMILES string for a chemistry model, a table for a table encoder. `Stream(key, stream)` moves `state[key]` into `record["streams"][stream]`. The rest of the state stays text, and the builder turns each stream into extra inputs of its own (see [Side stream inputs](#side-stream-inputs) below):

In [ ]:
#| export
@transform("Stream")
class Stream(Transform):
    "Move `state[key]` out of the text and into the side stream `stream` (named `key` by default), which the encoder reads on its own"
    def __init__(self, key:str, stream:str|None=None): store_attr(); self.stream = stream or key
    def __call__(self, record):
        s = record["state"]
        if not isinstance(s, dict) or s.get(self.key) is None: return record
        s = dict(s); v = s.pop(self.key)
        text = s.pop("text") if set(s) == {"text"} else (s if s else "")
        return {**record, "state": text, "streams": {**(record.get("streams") or {}), self.stream: v}}

In [ ]:
r = Stream("sequence", "protein")({"state": {"sequence": "MKTAYIAK", "organism": "Homo sapiens"}, "questions": {}})
test_eq((r["state"], r["streams"]), ({"organism": "Homo sapiens"}, {"protein": "MKTAYIAK"}))
test_eq(Stream("sequence")({"state": {"sequence": "MK", "text": "context"}, "questions": {}})["state"], "context")
test_eq(Stream("sequence")({"state": "plain text", "questions": {}})["state"], "plain text")

A transform of your own registers by name, so an export that uses it loads anywhere the class is importable:

In [ ]:
@transform("MaskEmails")
class MaskEmails(Transform):
    def __init__(self, repl="<email>"): store_attr()
    def __call__(self, record): return _map_record(record, lambda s: re.sub(r"\S+@\S+", self.repl, s), options=False)

test_eq(Transform.from_json({"name": "MaskEmails"})({"state": "from a@b.com", "questions": {}})["state"], "from <email>")

## RowTemplate

A `RowTemplate` is one format string plus the special tokens and the budgets. `row` may use `{start}`, `{sep}`, `{end}`, `{mask}` and `{image}` for the special tokens, `{type}` and `{instructions}` for the question, `{options}` for the options and `{state}` for the state; `option` is the text of one option, `{mask}` then `{text}`. Budgets are Laya's: every option at most `option_tokens` tokens after its anchor, question and options within `head_max_len`, the whole row within `max_len`, and the state gets what is left.

In [ ]:
#| export
LAYA_ROW = "{start}{type} question: {instructions}{sep}{options}{sep}{state}{end}"
_ROW_FIELDS = {"start", "sep", "end", "mask", "image", "type", "instructions", "options", "state"}
_OPTION_FIELDS = {"mask", "text"}
SPECIALS = ("start", "sep", "end", "mask", "image")

def _parse(fmt:str, allowed:set):
    "Split a format string into ('lit', text) and ('field', name) pieces"
    out = []
    for lit, name, spec, conv in string.Formatter().parse(fmt):
        if lit: out.append(("lit", lit))
        if name is None: continue
        if name not in allowed: raise ValueError(f"unknown field {{{name}}} in {fmt!r}; use {sorted(allowed)}")
        out.append(("field", name))
    return out

class RowTemplate:
    "How a question and a state are laid out as one row, which special tokens it uses, and the token budgets"
    def __init__(self,
                 start:str|None="[CLS]", # token that opens the row
                 sep:str|None="[SEP]", # token between sections
                 mask:str="[MASK]", # the anchor placed before every option; must be one token
                 end:str|None="[SEP]", # token that closes the row
                 image:str|None=None, # image placeholder token, for processor encoders
                 row:str=LAYA_ROW, # the row's format string
                 option:str="{mask} {text}", # one option's format string; must start with {mask}
                 max_len:int=512, # tokens per row, image positions included
                 head_max_len:int=192, # tokens for the question and its options
                 option_tokens:int=48, # text tokens per option
                 truncate:str="right", # cut long states on the "right", "left", or "auto" (left for conversation lists, as Laya does)
                 option_sep:str="", # text between options; tokenizer rows concatenate the option chunks
                 tfms:list|None=None, # transforms saved with the template
                 name:str|None=None): # preset name, for display
        store_attr(but="tfms"); self.tfms = list(tfms or [])
        self._row, self._option = _parse(row, _ROW_FIELDS), _parse(option, _OPTION_FIELDS)
        if ("field", "options") not in self._row: raise ValueError("a row needs an {options} field")
        if ("field", "state") not in self._row: raise ValueError("a row needs a {state} field")
        if not self._option or self._option[0] != ("field", "mask"):
            raise ValueError("an option must start with {mask}, so truncation never removes the anchor")
        if truncate not in ("right", "left", "auto"): raise ValueError(f"truncate must be 'right', 'left' or 'auto', not {truncate!r}")

    def __repr__(self):
        return (f"RowTemplate({self.name or 'custom'}: {self.row!r}, option={self.option!r}, mask={self.mask!r}, "
                f"budgets {self.max_len}/{self.head_max_len}/{self.option_tokens})")

In [ ]:
#| export
_TEMPLATE_KEYS = ("start", "sep", "end", "mask", "image", "row", "option", "max_len", "head_max_len",
                  "option_tokens", "truncate", "option_sep")

@patch
def to_json(self:RowTemplate) -> dict:
    "The template as it is written to `sysone.json` (transforms are written beside it)"
    return {k: getattr(self, k) for k in _TEMPLATE_KEYS} | ({"name": self.name} if self.name else {})

@patch(cls_method=True)
def from_json(cls:RowTemplate, d:dict, tfms=None):
    "The template described by a `to_json()` result"
    return cls(**{k: v for k, v in d.items() if k in _TEMPLATE_KEYS + ("name",)}, tfms=tfms)

@patch
def replace(self:RowTemplate, **kw):
    "A copy with some fields changed"
    return RowTemplate(**({k: getattr(self, k) for k in _TEMPLATE_KEYS} | {"tfms": self.tfms, "name": self.name} | kw))

@patch(as_prop=True)
def hash(self:RowTemplate) -> str:
    "A short digest of the template and its transforms, part of the feature-cache key"
    blob = json.dumps({"t": self.to_json() | {"name": None}, "tfms": tfms_to_json(self.tfms)}, sort_keys=True)
    return hashlib.sha1(blob.encode()).hexdigest()[:16]

### Presets

`laya` is read off ModernBERT's tokenizer and Laya's runtime, including its truncation rule (a conversation list keeps its end); given a tokenizer, it takes that tokenizer's cls, sep and mask tokens, as Laya's runtime does, so mmBERT's `<bos>`, `<eos>` and `<mask>` work unchanged. `gliner` is GLiNER2's own layout on the Decide tokenizer with its trained `[L]` as the anchor, and, as GLiNER2's processor builds it, no `[CLS]` or `[SEP]` around the row. `neomme` is NeoMME's: a `<doc>` prefix, `<mask>`, `<img>`, no separator tokens and nothing appended, so the sections and the options are separated by newlines, and each option is `<mask>` directly followed by its text (NeoMME's tokenizer merges a space into the next word, and its masks were trained in place of whole tokens); `neomme_page_first` puts the page, image and text, before the question, the layout NeoMME's pretraining saw for pages. `modernvbert` is Laya's layout on ModernBERT's `[CLS]`, `[SEP]` and `[MASK]`, with the image, ModernVBERT's `<image>` block, between the options and the state, and a longer row budget for the image's tokens (see [ModernVBERT](23_modernvbert.ipynb)). `auto` reads cls, sep and mask off a tokenizer and fails naming the tokens it could not find.

In [ ]:
#| export
_PRESETS = {
    "laya": dict(start="[CLS]", sep="[SEP]", mask="[MASK]", end="[SEP]", row=LAYA_ROW, option="{mask} {text}",
                 max_len=512, head_max_len=192, option_tokens=48, truncate="auto"),
    "gliner": dict(start=None, sep="[SEP_TEXT]", end=None, mask="[L]",
                   row="{start}( [P] {type}: {instructions} ( {options} ) ){sep}{state}{end}",
                   option="{mask} {text}", max_len=512, head_max_len=192, option_tokens=48, truncate="right"),
    "neomme": dict(start="<doc>", sep="\n", mask="<mask>", end=None, image="<img>",
                   row="{start}{type} question: {instructions}{sep}{options}{sep}{image}{state}{end}",
                   option="{mask}{text}", option_sep="\n", max_len=2048, head_max_len=384, option_tokens=48, truncate="right"),
    "neomme_page_first": dict(start="<doc>", sep="\n", mask="<mask>", end=None, image="<img>",
                   row="{start}{image}{state}{sep}{type} question: {instructions}{sep}{options}{end}",
                   option="{mask}{text}", option_sep="\n", max_len=2048, head_max_len=384, option_tokens=48, truncate="right"),
    "modernvbert": dict(start="[CLS]", sep="[SEP]", mask="[MASK]", end="[SEP]", image="<image>",
                        row="{start}{type} question: {instructions}{sep}{options}{sep}{image}{state}{end}",
                        option="{mask} {text}", max_len=2048, head_max_len=384, option_tokens=48, truncate="right"),
}

@patch(cls_method=True)
def preset(cls:RowTemplate, name:str, tokenizer=None, **overrides):
    "A named template (`laya`, `gliner`, `neomme`, `neomme_page_first`, `modernvbert`, `auto`), optionally with fields changed"
    if name == "auto" and tokenizer is None:
        raise ValueError("RowTemplate.preset('auto') needs the tokenizer to read its special tokens from")
    if name in ("laya", "auto") and tokenizer is not None:   # Laya's runtime reads cls, sep and mask off the tokenizer
        tok = getattr(tokenizer, "tokenizer", tokenizer)
        found = {r: getattr(tok, f"{r}_token", None) for r in ("cls", "sep", "mask")}
        missing = [f"{r}_token" for r, v in found.items() if not v]
        if missing and name == "auto": raise ValueError(f"the tokenizer defines no {', '.join(missing)}; name a RowTemplate for this encoder")
        kw = _PRESETS["laya"] | ({} if missing else dict(start=found["cls"], sep=found["sep"], end=found["sep"], mask=found["mask"]))
    elif name in _PRESETS: kw = dict(_PRESETS[name])
    else: raise ValueError(f"unknown preset {name!r}; use one of {sorted(_PRESETS) + ['auto']}")
    return cls(**(kw | overrides), name=name)

In [ ]:
laya = RowTemplate.preset("laya")
laya

RowTemplate(laya: '{start}{type} question: {instructions}{sep}{options}{sep}{state}{end}', option='{mask} {text}', mask='[MASK]', budgets 512/192/48)

In [ ]:
test_eq(RowTemplate.from_json(laya.to_json()).to_json(), laya.to_json())
test_eq(RowTemplate.preset("laya", mask="[L]").mask, "[L]")
test_ne(RowTemplate.preset("laya", mask="[L]").hash, laya.hash)
test_eq(RowTemplate.preset("auto", tokenizer=tok).to_json() | {"name": "laya"}, laya.to_json())
test_eq(RowTemplate.preset("laya", tokenizer=tok).to_json(), laya.to_json())
test_fail(lambda: RowTemplate(row="{start}{options}"), contains="{state}")
test_fail(lambda: RowTemplate(option="{text} {mask}"), contains="start with {mask}")
test_fail(lambda: RowTemplate(row="{start}{question}{options}{state}"), contains="unknown field")

## RowBuilder

`RowBuilder(tokenizer, template, tfms)` turns a record into rows. The row format compiles into segments: special tokens, *runs* of literal text and the `{type}`/`{instructions}` fields, the options, and the state. Each segment is tokenized on its own and the ids are concatenated, so marker positions are exact and the budgets apply per segment:

1. every option is its anchor id followed by at most `option_tokens` text tokens;
2. if the options leave the question fewer than 16 of the `head_max_len` tokens, every option is cut to an equal share (at least 4 tokens, anchor first, so the anchor survives);
3. the run holding the instructions gets whatever `head_max_len` has left, at least 8 tokens;
4. the state gets the rest of `max_len`, cut on the right, or on the left with `truncate="left"`.

These are Laya's `build_sequence` rules, so on Laya's template the rows are identical to Laya's, token for token (tested below against the `laya` package). A special token that is not a single token of the vocabulary, such as NeoMME's newline separator, is tokenized as text. The anchor is scrubbed from user text so a stray `[MASK]` in a state cannot become a marker.

In [ ]:
#| export
def _compile(pieces):
    "Group pieces into segments: ('special', role), ('run', [pieces]), ('options',), ('state',)"
    segs, run = [], []
    def flush():
        nonlocal run
        if run: segs.append(("run", run)); run = []
    for kind, v in pieces:
        if kind == "field" and v in SPECIALS: flush(); segs.append(("special", v))
        elif kind == "field" and v in ("options", "state"): flush(); segs.append((v,))
        else: run.append((kind, v))
    flush()
    return segs

def _is_processor(obj):
    "Whether `obj` is a transformers processor (tokenizer plus image processor) rather than a tokenizer"
    from transformers import ProcessorMixin
    return isinstance(obj, ProcessorMixin)

class RowBuilder:
    "Build rows from records with a tokenizer (or a processor, for multimodal encoders) and a `RowTemplate`"
    def __init__(self, tokenizer, template:RowTemplate|str="laya", tfms:list|None=None, streams=None):
        if isinstance(template, str): template = RowTemplate.preset(template, tokenizer=tokenizer)
        self.processor = tokenizer if _is_processor(tokenizer) else None
        self.tok = tokenizer.tokenizer if self.processor is not None else tokenizer
        self.template, self.tfms = template, list(template.tfms) + list(tfms or [])
        self._row_segs, self._opt_segs = _compile(template._row), _compile(template._option)
        self.ids = self._resolve_specials()
        self.extra_scrub = []   # further token strings to remove from user text (a multimodal encoder's own markers)
        self.streams = {s.name: s for s in (streams.values() if isinstance(streams, dict) else streams or [])}
        self.stats = {"rows": 0, "state_truncated": 0, "options_cut": 0, "dropped": 0}

    def __repr__(self):
        kind = "processor" if self.processor is not None else "tokenizer"
        streams = f", streams={list(self.streams.values())}" if self.streams else ""
        return f"RowBuilder({type(self.tok).__name__} {kind}, {self.template!r}, tfms={self.tfms}{streams})"

In [ ]:
#| export
def _single_token_id(tok, s):
    "The id of `s` if the tokenizer knows it as exactly one token, else None"
    if not s: return None
    tid = tok.convert_tokens_to_ids(s)
    if tid is None or (tid == tok.unk_token_id and s != tok.unk_token): return None
    return tid if tok(s, add_special_tokens=False)["input_ids"] == [tid] else None

@patch
def _resolve_specials(self:RowBuilder) -> dict:
    "Token ids of the template's special tokens; the anchor must be one token"
    t, ids = self.template, {}
    for role in SPECIALS:
        s = getattr(t, role)
        ids[role] = _single_token_id(self.tok, s) if s else None
    if ids["mask"] is None:
        raise ValueError(f"the anchor {t.mask!r} is not a single token of this tokenizer; pick a token it knows "
                         f"(its mask token is {getattr(self.tok, 'mask_token', None)!r})")
    return ids

In [ ]:
builder = RowBuilder(tok, "laya")
test_eq(builder.ids, {"start": tok.cls_token_id, "sep": tok.sep_token_id, "end": tok.sep_token_id, "mask": tok.mask_token_id, "image": None})
test_fail(lambda: RowBuilder(tok, RowTemplate.preset("laya", mask="[L]")), contains="not a single token")
builder

RowBuilder(TokenizersBackend tokenizer, RowTemplate(laya: '{start}{type} question: {instructions}{sep}{options}{sep}{state}{end}', option='{mask} {text}', mask='[MASK]', budgets 512/192/48), tfms=[])

### Tokenizing a record

`_texts` collects everything a record needs tokenized, the state once and each question's runs and options, so one batched tokenizer call serves the whole case. The helpers below render a run for a question, find the part of a run that may be cut (the literal text after the field is kept), and cut it:

In [ ]:
#| export
@patch
def _scrub(self:RowBuilder, s:str) -> str:
    "User text with the anchor and image tokens replaced by spaces"
    for tokstr in [getattr(self.template, r) for r in ("mask", "image")] + self.extra_scrub:
        if tokstr: s = s.replace(tokstr, " ")
    return s

def _render_run(parts, q=None, text=None):
    "The text of a run, and the literal text after its variable field (kept when the run is cut)"
    out, tail, seen = [], [], False
    for kind, v in parts:
        s = v if kind == "lit" else (q.type if v == "type" else q.instructions if v == "instructions" else text)
        out.append(s)
        if kind == "field" and v in ("instructions", "text"): seen, tail = True, []
        elif seen: tail.append(s)
    return "".join(out), "".join(tail)

def _cut(ids, n, tail_ids):
    "At most n ids, keeping the ids of the run's literal tail"
    if len(ids) <= n: return ids
    return ids[:max(0, n - len(tail_ids))] + list(tail_ids)

In [ ]:
test_eq(_render_run([("field", "type"), ("lit", " question: "), ("field", "instructions")], Question("noul", "Is it?")), ("noul question: Is it?", ""))
test_eq(_render_run([("lit", "( [P] "), ("field", "type"), ("lit", ": "), ("field", "instructions"), ("lit", " ( ")], Question("noul", "Is it?")),
        ("( [P] noul: Is it? ( ", " ( "))
test_eq(_cut([1, 2, 3, 4, 5, 9], 4, [9]), [1, 2, 3, 9])
test_eq(_cut([1, 2, 3], 4, [9]), [1, 2, 3])

In [ ]:
#| export
@patch
def _encode(self:RowBuilder, texts:list) -> list:
    "Token ids of several texts in one tokenizer call; the empty string has none"
    idx = [i for i, s in enumerate(texts) if s]   # verbose=False: a long state is expected here, the layout truncates it
    enc = self.tok([texts[i] for i in idx], add_special_tokens=False, verbose=False)["input_ids"] if idx else []
    out = [[] for _ in texts]
    for i, ids in zip(idx, enc): out[i] = list(ids)
    return out

@patch
def _question_parts(self:RowBuilder, q:Question):
    "Texts to tokenize for one question: each run of the row, each option's runs"
    q = Question(q.type, self._scrub(q.instructions), q.criteria, q.labels, q.name)
    runs = [_render_run(s[1], q) for s in self._row_segs if s[0] == "run"]
    opts = [[_render_run(s[1], q, self._scrub(o)) for s in self._opt_segs if s[0] == "run"] for o in q.options]
    return runs, opts

### Assembling a row

`_assemble` applies the budgets to one question's tokenized pieces and lays the segments out in template order, recording where each option's anchor lands:

In [ ]:
#| export
@patch
def _assemble(self:RowBuilder, q:Question, run_ids, opt_ids, state_ids, state_is_list=False, image_ids=None):
    "ids, marker positions and the image marker's position for one question, with Laya's budget rules"
    t, image_ids = self.template, list(image_ids or [])
    # 1. options: anchor + at most option_tokens text tokens (per text run, literal tail kept)
    chunks = []
    for runs in opt_ids:
        c, r = [], iter(runs)
        for seg in self._opt_segs:
            if seg[0] == "special": c.append(self.ids[seg[1]])
            elif seg[0] == "run": ids, tail = next(r); c.extend(_cut(ids, t.option_tokens, tail))
        chunks.append(c)
    sep_ids = self._encode([t.option_sep])[0] if t.option_sep else []
    opt_len = lambda: sum(len(c) for c in chunks) + len(sep_ids) * max(0, len(chunks) - 1)
    # 2. squeeze the options if they leave the question fewer than 16 tokens
    budget = t.head_max_len - opt_len()
    if budget < 16:
        per = max(4, (t.head_max_len - 16) // max(1, len(chunks)))
        if any(len(c) > per for c in chunks): self.stats["options_cut"] += 1
        chunks = [c[:per] for c in chunks]
        budget = t.head_max_len - opt_len()
    # 3. the run holding the instructions gets what head_max_len has left
    runs = []
    for (ids, tail), seg in zip(run_ids, [s for s in self._row_segs if s[0] == "run"]):
        has_ins = ("field", "instructions") in seg[1]
        runs.append(_cut(ids, max(8, budget), tail) if has_ins else ids)
    # 4. the state gets the rest of max_len
    specials = lambda role: [] if getattr(t, role) is None else [self.ids[role]] if self.ids[role] is not None else self._encode([getattr(t, role)])[0]
    img_seg = [i for i, s in enumerate(self._row_segs) if s == ("special", "image")]
    if image_ids and not img_seg: raise ValueError("the record has an image, but the template has no {image} field")
    dedupe = bool(image_ids) and img_seg[0] > 0 and self._row_segs[img_seg[0] - 1] == ("special", "start") \
             and self.ids["start"] == image_ids[0]      # the block's leading token already opens the row
    fixed = sum(len(specials(s[1])) for s in self._row_segs if s[0] == "special" and s[1] != "image")
    fixed += sum(len(r) for r in runs) + opt_len() + len(image_ids) - int(dedupe)
    room = max(0, t.max_len - fixed)
    left = t.truncate == "left" or (t.truncate == "auto" and state_is_list)
    if len(state_ids) > room: self.stats["state_truncated"] += 1
    st = state_ids[max(0, len(state_ids) - room):] if left else state_ids[:room]
    # lay the segments out in template order
    ids, markers, spans, r, image_at = [], [], [], iter(runs), None
    for seg in self._row_segs:
        if seg[0] == "special":
            if seg[1] != "image": ids.extend(specials(seg[1]))
            elif image_ids:
                block = image_ids[1:] if ids and ids[-1] == image_ids[0] else image_ids
                image_at = len(ids) + len(block) - len(image_ids) + 1     # the image marker, after its opening token
                ids.extend(block)
        elif seg[0] == "run": ids.extend(next(r))
        elif seg[0] == "state": ids.extend(st)
        else:
            for i, c in enumerate(chunks):
                if i and sep_ids: ids.extend(sep_ids)
                markers.append(len(ids)); spans.append((len(ids) + 1, len(ids) + len(c))); ids.extend(c)
    if image_ids and len(ids) > t.max_len:
        raise ValueError(f"the image takes {len(image_ids)} positions and the row does not fit max_len={t.max_len}; resize the image")
    keep = [i for i, m in enumerate(markers) if m < t.max_len]
    return ids[:t.max_len], [markers[i] for i in keep], image_at, [(s, min(e, t.max_len)) for s, e in (spans[i] for i in keep)]

### Building rows

`build_record` runs the transforms over a whole case, tokenizes it with one call (`_layout`), and returns one `Row` per question, each carrying the target distribution and label when the record has gold. A question whose options do not fit the budget is an error at inference and is skipped, with a count, when building training data. `build` is the one-question convenience.

A builder on a processor hands the record to `processor_rows`, a generic function that dispatches on the processor's class (see [generic functions](00_core.ipynb#generic-functions-and-multiple-dispatch)). Each multimodal encoder lays an image out its own way (NeoMME as a patch grid with two-axis positions, ModernVBERT as tiles of `<image>` tokens), while the text around it follows the template's budgets as for any row. Two more generic functions on the processor tell the [multimodal application](20_neomme.ipynb#a-head-only-run) what an encoder's images need. `default_image_side(processor)` is the side they are resized to unless the caller says otherwise, and `prepare_processor(processor, side)` sets the processor up for that side and says how many images each row then carries (a tiling processor makes several of one). Their fallbacks import sysone's own adapter for the class on first use. Without an adapter, `processor_rows` fails with a message, and the image defaults are the neutral ones: images as they are, one per row.

In [ ]:
#| export
@dispatch
def processor_rows(processor:object, builder, record:dict, *, skip_errors:bool=False) -> list:
    "The rows of a transformed record for a multimodal encoder: dispatched on its processor's class, whose adapter adds the method"
    if load_adapter(processor): return processor_rows(processor, builder, record, skip_errors=skip_errors)
    raise ValueError(f"no row builder for a {type(processor).__name__}: register one with `processor_rows.register`")

@dispatch
def default_image_side(processor:object):
    "The side, in pixels, an encoder's images are resized to by default: dispatched on its processor's class (None: as they are)"
    if load_adapter(processor): return default_image_side(processor)
    return None

@dispatch
def prepare_processor(processor:object, side:int|None) -> int:
    "Set a processor up for images of `side` pixels: how many images each row then carries (1, unless the processor tiles)"
    if load_adapter(processor): return prepare_processor(processor, side)
    return 1

@patch
def build_record(self:RowBuilder, record:dict, train:bool=False, skip_errors:bool=False, tfms:bool=True) -> list:
    "One `Row` per question of a record (a case with `state`, `questions`, optional `gold`)"
    record = load_record(record, parse_state=False)
    if tfms: record = apply_tfms(self.tfms, record, train=train)
    if self.processor is not None: return self._add_streams(record, processor_rows(self.processor, self, record, skip_errors=skip_errors))
    gold, rows = record.get("gold") or {}, []
    for qid, q, ids, markers, _, spans in self._layout(record, skip_errors=skip_errors):
        target, label = q.target(gold.get(qid))
        rows.append(Row(ids, markers, q.qtype, target, label, meta={"qid": qid, "case": record.get("id")},
                        spans=spans, answerable=q.answerable(gold.get(qid))))
    return self._add_streams(record, rows)

@patch
def _add_streams(self:RowBuilder, record:dict, rows:list) -> list:
    "Every row of the record gets the inputs of each side stream (a record without one gets the stream's empty input)"
    if not self.streams or not rows: return rows
    vals, ex = record.get("streams") or {}, {}
    for name, s in self.streams.items(): ex |= s.build(vals.get(name))
    for r in rows: r.extras.update(ex)
    return rows

@patch
def _layout(self:RowBuilder, record:dict, image_ids=None, skip_errors:bool=False) -> list:
    "(qid, question, ids, markers, image marker position, option spans) for each question of a transformed record"
    qs = as_questions(record["questions"])
    parts = {qid: self._question_parts(q) for qid, q in qs.items()}
    texts = [self._scrub(render_state(record["state"]))]
    for runs, opts in parts.values():
        for s, tail in runs: texts += [s, tail]
        for o in opts:
            for s, tail in o: texts += [s, tail]
    enc = iter(self._encode(texts))
    state_ids = next(enc)
    out = []
    for qid, q in qs.items():
        runs, opts = parts[qid]
        run_ids = [(next(enc), next(enc)) for _ in runs]
        opt_ids = [[(next(enc), next(enc)) for _ in o] for o in opts]
        ids, markers, image_at, spans = self._assemble(q, run_ids, opt_ids, state_ids, isinstance(record["state"], list), image_ids)
        self.stats["rows"] += 1
        if len(markers) != q.k:
            if not skip_errors: raise ValueError(f"question {qid!r}: its {q.k} options do not fit max_len={self.template.max_len}")
            self.stats["dropped"] += 1; continue
        out.append((qid, q, ids, markers, image_at, spans))
    return out

@patch
def build(self:RowBuilder, state, question, gold=None, name="q") -> Row:
    "The row for one question over one state"
    q = Question.from_dict(question, name=name)
    return self.build_record({"state": state, "questions": {name: q.to_dict()}, "gold": {name: gold}})[0]

In [ ]:
class _Unknown: pass
test_fail(lambda: processor_rows(_Unknown(), None, {}), contains="no row builder for a _Unknown")
test_eq((default_image_side(_Unknown()), prepare_processor(_Unknown(), 512)), (None, 1))       # the neutral defaults

A processor sysone has never met gets rows as soon as a method for its class is registered, which is all an encoder's adapter does:

In [ ]:
class _Toy: "a processor from a library sysone doesn't know"

def _toy_rows(processor:_Toy, builder, record:dict, *, skip_errors:bool=False) -> list: return [f"a row for {record['id']}"]
processor_rows.register(_toy_rows)
test_eq(processor_rows(_Toy(), None, {"id": "r1"}), ["a row for r1"])
methods(processor_rows)

[('object', 'Any', 'dict'), ('_Toy', 'Any', 'dict')]

In [ ]:
rows = builder.build_record(rec)
[(r.meta["qid"], r.n_tokens, r.markers) for r in rows]

[('action', 163, [20, 31, 45, 60]),
 ('needs_review', 129, [15, 24]),
 ('outcome', 170, [18, 30, 48, 65]),
 ('risk', 178, [21, 36, 48, 64]),
 ('urgency', 168, [18, 31, 44, 59])]

In [ ]:
r0 = rows[0]
test_eq([r0.ids[m] for m in r0.markers], [tok.mask_token_id] * 4)
test_eq((r0.ids[0], r0.ids[-1]), (tok.cls_token_id, tok.sep_token_id))
test_eq(r0.label, 1)  # human_review
test_eq(builder.build(rec["state"], rec["questions"]["action"], rec["gold"]["action"]).ids, r0.ids)
test_eq([r0.ids[m] for m in r0.markers], [r0.ids[s - 1] for s, e in r0.spans])   # each option's tokens start right after its anchor
test_eq(r0.answerable, 1)

Each row also records the span of every option's own tokens, from just after its anchor to the end of its text: a head can read an option as the mean of those tokens instead of at the anchor, which helps when the encoder stays frozen and its anchor token was never trained to summarise what follows it (see [models](03_models.ipynb#readouts-and-queries)):

In [ ]:
s, e = r0.spans[1]
tok.decode(r0.ids[s:e])

' human_review: Queue this trace for a human to review.'

### Side stream inputs

A side stream moved out of the state by `Stream` becomes inputs of its own, which the encoder reads next to the row's ids. `TokenStream` tokenizes it with its own tokenizer, adds that tokenizer's special tokens and crops it to `max_len` tokens. It crops on the `right` (keep the start), on the `left` (keep the end), or at the `ends` (keep the start and the end, dropping the middle), which suits inputs whose informative parts sit at both extremes, such as the targeting signals at a protein's two termini. Its ids and attention mask are extras of every row of the record, collated with the tokenizer's own pad id, so several questions about one protein share one protein input.

```{mermaid}
flowchart LR
    R["record: state = {sequence, text}"] --> S["Stream('sequence', 'protein')"]
    S --> T["state text → the row<br/>(question, anchors, options)"]
    S --> P["streams['protein'] →<br/>TokenStream: its own tokenizer, cropped"]
    T --> X["input_ids, markers, spans"]
    P --> Y["protein_input_ids,<br/>protein_attention_mask"]
    X --> B["one Row per question"]
    Y --> B
```

In [ ]:
#| export
class TokenStream:
    "A side stream tokenized by its own tokenizer; its ids and attention mask ride in every row of the record"
    def __init__(self,
                 name:str, # the stream, as `Stream` names it; its inputs are `{name}_input_ids` and `{name}_attention_mask`
                 tokenizer, # the stream's tokenizer
                 max_len:int=1024, # tokens, special tokens included
                 crop:str="right"): # "right" keeps the start, "left" the end, "ends" the start and the end
        if crop not in ("right", "left", "ends"): raise ValueError(f"crop must be 'right', 'left' or 'ends', not {crop!r}")
        store_attr(); self.stats = {"built": 0, "cropped": 0}
        self.n_special = len(tokenizer("")["input_ids"])
        if max_len <= self.n_special: raise ValueError(f"max_len={max_len} leaves no room beside {self.n_special} special tokens")
        EXTRA_COLLATE[f"{name}_input_ids"] = ("pad_rows", tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0)
        EXTRA_COLLATE[f"{name}_attention_mask"] = ("pad_rows", 0)

    def __repr__(self): return f"TokenStream({self.name}: {type(self.tokenizer).__name__}, max_len={self.max_len}, crop={self.crop!r})"
    @property
    def columns(self) -> tuple: return (f"{self.name}_input_ids", f"{self.name}_attention_mask")
    def to_json(self) -> dict: return {"kind": "tokens", "name": self.name, "max_len": self.max_len, "crop": self.crop}

    def build(self, value) -> dict:
        "The stream's inputs for one record (just the special tokens when the record has none)"
        text = "" if value is None else str(value)
        full = self.tokenizer(text)["input_ids"]                                  # with the tokenizer's own special tokens
        ids = self.tokenizer(text, add_special_tokens=False)["input_ids"] if text else []
        at = next((i for i in range(len(full) - len(ids) + 1) if full[i:i + len(ids)] == ids), 0)
        head, tail, room = full[:at], full[at + len(ids):], self.max_len - (len(full) - len(ids))
        self.stats["built"] += 1
        if len(ids) > room:
            self.stats["cropped"] += 1
            ids = ids[:room] if self.crop == "right" else ids[-room:] if self.crop == "left" else ids[:room - room // 2] + ids[len(ids) - room // 2:]
        ids = head + ids + tail
        return {self.columns[0]: ids, self.columns[1]: [1] * len(ids)}

def stream_from_json(d:dict, tokenizer):
    "The side stream described by `to_json()`, on `tokenizer`"
    if d.get("kind", "tokens") != "tokens": raise ValueError(f"unknown stream kind {d.get('kind')!r}")
    return TokenStream(d["name"], tokenizer, d.get("max_len", 1024), d.get("crop", "right"))

Any tokenizer can serve a stream; the fixture's text tokenizer stands in for a protein tokenizer here. Two questions over one record share one stream input, and a record without the stream gets only the special tokens:

In [ ]:
notes = TokenStream("notes", tok, max_len=12, crop="ends")
sb = RowBuilder(tok, "laya", tfms=[Stream("notes")], streams=[notes])
srec = {"state": {"text": "Customer asks for a refund.", "notes": "the order number is 4411 and it was charged twice last week"},
        "questions": {"dept": {"type": "choice", "instructions": "Which team?", "criteria": ["billing", "support"]},
                      "urgent": {"type": "noul", "instructions": "Is it urgent?"}}}
srows = sb.build_record(srec)
test_eq(len(srows), 2)
test_eq(srows[0].extras["notes_input_ids"], srows[1].extras["notes_input_ids"])
ids = srows[0].extras["notes_input_ids"]
test_eq((len(ids), ids[0], ids[-1], notes.stats["cropped"]), (12, tok.cls_token_id, tok.sep_token_id, 1))
full = tok("the order number is 4411 and it was charged twice last week", add_special_tokens=False)["input_ids"]
test_eq(ids[1:-1], full[:5] + full[-5:])                        # "ends": the start and the end, the middle dropped
test_eq(sb.build_record({**srec, "state": {"text": "no notes"}})[0].extras["notes_input_ids"], [tok.cls_token_id, tok.sep_token_id])
test_eq(collate(srows)["notes_input_ids"].shape, (2, 12))
sb

RowBuilder(TokenizersBackend tokenizer, RowTemplate(laya: '{start}{type} question: {instructions}{sep}{options}{sep}{state}{end}', option='{mask} {text}', mask='[MASK]', budgets 512/192/48), tfms=[Stream(key='notes', stream='notes')], streams=[TokenStream(notes: TokenizersBackend, max_len=12, crop='ends')])

### Showing rows

`show_row` decodes a row with its anchors visible and the state shortened, fastai's `show_batch` habit: the quickest way to catch a wrong separator or a lost anchor, which would otherwise silently ruin a model. Runs of one repeated token, such as the patch placeholders of an image, are collapsed to `token×n`.

In [ ]:
#| export
def _decode_ids(tok, ids):
    specials = set(getattr(tok, "all_special_ids", []))
    out, buf = [], []
    def flush():
        if buf: out.append(tok.decode(buf).strip()); buf.clear()
    i = 0
    while i < len(ids):
        t = ids[i]
        if t in specials:
            flush(); j = i
            while j < len(ids) and ids[j] == t: j += 1
            s = tok.convert_ids_to_tokens(t)
            out.append(s if j - i == 1 else f"{s}×{j - i}"); i = j
        else: buf.append(t); i += 1
    flush()
    return " ".join(o for o in out if o)

def show_row(row:Row, tok, max_chars:int=300) -> str:
    "A row as readable text, anchors visible, plus its markers, type and length"
    tok = getattr(tok, "tokenizer", tok) if not hasattr(tok, "convert_ids_to_tokens") else tok
    s = _decode_ids(tok, list(row.ids))
    if len(s) > max_chars: s = s[:max_chars * 2 // 3] + " … " + s[-max_chars // 3:]
    streams = "".join(f" · {k.removesuffix('_input_ids')} {len(v)} tokens" for k, v in row.extras.items() if k.endswith("_input_ids"))
    return f"{s}\nmarkers {list(row.markers)} · qtype {QTYPE_NAMES[row.qtype]} · {row.n_tokens} tokens{streams}"

@patch
def show(self:RowBuilder, record:dict, max_chars:int=300):
    "Print the rows of a record"
    for r in self.build_record(record): print(show_row(r, self.tok, max_chars), end="\n\n")

In [ ]:
builder.show(rec)

[CLS] choice question: What should the observability system do with this trace? [SEP] [MASK] continue: Let the agent proceed without interruption. [MASK] human_review: Queue this trace for a human to  … violations": 0, "duration_s": 128.5, "irreversible_actions": 0, "steps": 7, "tool_errors": 1}} [SEP]
markers [20, 31, 45, 60] · qtype choice · 163 tokens

[CLS] noul question: This trace requires human review. [SEP] [MASK] false: No human attention is warranted. [MASK] true: A human should inspect this run. [SEP] {"agent": {"autonomy": "checkpointed", " … violations": 0, "duration_s": 128.5, "irreversible_actions": 0, "steps": 7, "tool_errors": 1}} [SEP]
markers [15, 24] · qtype noul · 129 tokens

[CLS] choice question: How did this agent run turn out? [SEP] [MASK] failure: The agent did not accomplish the task. [MASK] harmful: The agent took an action that caused damage or violated a constrain … violations": 0, "duration_s": 128.5, "irreversible_actions": 0, "steps": 7, "tool_errors": 

### Budgets

On a real row the budgets rarely bind: typed-decisions' longest state is about 2,000 characters. They matter for long pages and wide option sets. Here the whole row is squeezed into 64 tokens, and the fifteen-option question has to share its option budget:

In [ ]:
small = RowBuilder(tok, laya.replace(max_len=96, head_max_len=48, option_tokens=6))
r = small.build(rec["state"], rec["questions"]["action"])
test_eq(r.n_tokens, 96)
test_eq([b - a for a, b in zip(r.markers, r.markers[1:])], [7, 7, 7])      # anchor + 6 tokens each
test_eq(small.stats["options_cut"], 0)
wide = {"type": "choice", "instructions": "Which one?", "criteria": {f"option {i}": "a long description of it" for i in range(15)}}
rw = small.build("state", wide)
test_eq((rw.k, small.stats["options_cut"]), (15, 1))
test_eq([b - a for a, b in zip(rw.markers, rw.markers[1:])], [4] * 14)   # an equal share, anchor first
print(show_row(r, tok)); print(show_row(rw, tok))

[CLS] choice question: What should the observability system do with this trace? [SEP] [MASK] continue: Let the agent [MASK] human_review: Queue this [MASK] observe: Keep running, [MASK] stop: Halt the … uch customer data outside the named accounts"], "task": "Delete personal data for the accounts [SEP]
markers [20, 27, 34, 41] · qtype choice · 96 tokens
[CLS] choice question: W [SEP] [MASK] option 0 [MASK] option 1 [MASK] option 2 [MASK] option 3 [MASK] option 4 [MASK] option 5 [MASK] option 6 [MASK] option 7 [MASK] option 8 [MASK] option 9 [MASK] option 10 [MASK] option 11 [MASK] option 12 [MASK] option 1 [MASK] option 14 [SEP] state [SEP]
markers [10, 14, 18, 22, 26, 30, 34, 38, 42, 46, 50, 54, 58, 62, 66] · qtype choice · 74 tokens


Drawn token by token, a row whose budgets all bind (40 tokens in all, 24 for the question and its options, 3 per option): each option keeps its anchor and three tokens, the question gets what the options leave, losing its question mark, and the state gets the rest of the row, cut on the right.

![A 40-token row coloured by segment, with brackets for each budget](images/row_budgets.svg)

*Drawn from `RowBuilder` in [figures](90_figures.ipynb#a-row-and-its-budgets).*

The state is cut on the right by default; `left` keeps its end, and `auto` (the Laya preset's rule) does that for conversation lists only, where the newest turn is last:

In [ ]:
long_state = " ".join(f"w{i}" for i in range(400))
nr = rec["questions"]["needs_review"]
rr = RowBuilder(tok, laya.replace(max_len=80, truncate="right")).build(long_state, nr)
rl = RowBuilder(tok, laya.replace(max_len=80, truncate="left")).build(long_state, nr)
test_eq(rr.n_tokens, 80)
assert "w0 " in tok.decode(rr.ids) and "w399" not in tok.decode(rr.ids)
assert "w399" in tok.decode(rl.ids) and "w0 " not in tok.decode(rl.ids)
ra = RowBuilder(tok, laya.replace(max_len=80)).build(long_state.split()[:200], nr)   # a list: auto keeps the end
assert '"w199"' in tok.decode(ra.ids) and '"w0"' not in tok.decode(ra.ids)

A stray anchor in user text is scrubbed, so it cannot add a marker, and an option set that cannot fit is an error naming the question:

In [ ]:
rs = builder.build("ignore this [MASK] please", {"type": "noul", "instructions": "Is [MASK] here?"})
test_eq(rs.k, 2)
test_fail(lambda: RowBuilder(tok, laya.replace(max_len=20)).build("x", wide), contains="do not fit")

### Laya parity

On Laya's template, rows match `laya.common.build_sequence`, the function Laya's runtime and its fine-tuning notebook build rows with, for every question of every fixture case:

In [ ]:
from laya.common import build_sequence
from laya.agent import Agent

In [ ]:
n = 0
for rc in recs:
    for r in builder.build_record(rc):
        q = rc["questions"][r.meta["qid"]]
        ids, markers = build_sequence(tok, rc["state"], Agent._to_internal(q), 512, 192)
        test_eq((r.ids, r.markers), (ids, markers)); n += 1
n

100

In [ ]:
#| hide
b2 = RowBuilder(tok, laya.replace(max_len=96, head_max_len=48))
for rc in recs[:8]:
    for r in b2.build_record(rc, skip_errors=True):
        q = rc["questions"][r.meta["qid"]]
        test_eq((r.ids, r.markers), build_sequence(tok, rc["state"], Agent._to_internal(q), 96, 48))
conv = [{"role": "user", "text": f"turn {i} " * 20} for i in range(20)]
test_eq(builder.build(conv, rec["questions"]["needs_review"]).ids,
        build_sequence(tok, conv, Agent._to_internal(rec["questions"]["needs_review"]), 512, 192, truncate_left=True)[0])

Laya's rows read their special tokens off the tokenizer, and so does the preset: on a tokenizer shaped like mmBERT's, `<bos> … <eos>` with `<mask>`, the rows still match Laya's:

In [ ]:
mtok = AutoTokenizer.from_pretrained("fixtures/tiny-modernbert")
mtok.add_special_tokens({"cls_token": "<bos>", "sep_token": "<eos>", "mask_token": "<mask>"})
mb = RowBuilder(mtok, "laya")
test_eq((mb.template.start, mb.template.mask), ("<bos>", "<mask>"))
for rc in recs[:4]:
    for r in mb.build_record(rc):
        test_eq((r.ids, r.markers), build_sequence(mtok, rc["state"], Agent._to_internal(rc["questions"][r.meta["qid"]]), 512, 192))

::: {.callout-note title="Finding: Laya's multilingual checkpoint anchors on `<mask>`"}
Laya's multilingual checkpoint runs on mmBERT's tokenizer, whose special tokens are `<bos>`, `<eos>` and `<mask>`, and Laya's runtime reads them off the tokenizer. The first version of the `laya` preset wrote ModernBERT's `[CLS]`, `[SEP]` and `[MASK]` into the template, so on that checkpoint the builder refused to start: `[MASK]` is not a token of mmBERT's vocabulary. The preset now takes the tokenizer's own tokens, and `Decider.load("convaiinnovations/laya/multilingual")` gives the same probabilities as `laya.Agent` on 153 of 153 decisions (30 typed-decisions test cases and a Spanish one).
:::

### Serialising a builder

A builder is written to `sysone.json` as its template, transforms and side streams, and rebuilt from them with the tokenizers saved beside it:

In [ ]:
#| export
@patch
def to_json(self:RowBuilder) -> dict:
    "The template, transforms and side streams, as written to `sysone.json`"
    return {"template": self.template.to_json(), "tfms": tfms_to_json(self.tfms)} | \
           ({"streams": [s.to_json() for s in self.streams.values()]} if self.streams else {})

@patch(cls_method=True)
def from_json(cls:RowBuilder, d:dict, tokenizer, stream_tokenizers:dict|None=None):
    "The builder described by `to_json()`, on `tokenizer` (and each side stream on its tokenizer in `stream_tokenizers`)"
    streams = [stream_from_json(s, (stream_tokenizers or {})[s["name"]]) for s in d.get("streams") or []]
    return cls(tokenizer, RowTemplate.from_json(d["template"]), tfms_from_json(d.get("tfms")), streams=streams)

In [ ]:
b3 = RowBuilder(tok, "laya", tfms=[Regex(r"https?://\S+", "<url>")])
b4 = RowBuilder.from_json(json.loads(json.dumps(b3.to_json())), tok)
test_eq(b4.build_record(rec)[2].ids, b3.build_record(rec)[2].ids)
test_eq(tfms_to_json(b4.tfms), tfms_to_json(b3.tfms))
sb2 = RowBuilder.from_json(json.loads(json.dumps(sb.to_json())), tok, {"notes": tok})
test_eq(sb2.build_record(srec)[0].extras, srows[0].extras)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()